# Feature-Aligned Tuning with Dynamic-K

Evaluation includes both fixed `Top-11` and `dynamic-k` metrics under user-grouped nested CV.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss, precision_score, recall_score, f1_score

warnings.filterwarnings("ignore")

try:
    import lightgbm as lgb
    HAS_LIGHTGBM = True
except Exception:
    HAS_LIGHTGBM = False

print("HAS_LIGHTGBM:", HAS_LIGHTGBM)

HAS_LIGHTGBM: True


In [3]:
# ----------------------------
# Config
# ----------------------------
DATA_DIR = Path("data")
RANDOM_STATE = 42

# For speed while iterating. Set None for full users.
SAMPLE_USERS = 20000

OUTER_SPLITS = 3
INNER_SPLITS = 3
TOP_K_FIXED = 11
N_ITER_SEARCH = 12

print({
    "DATA_DIR": str(DATA_DIR),
    "SAMPLE_USERS": SAMPLE_USERS,
    "OUTER_SPLITS": OUTER_SPLITS,
    "INNER_SPLITS": INNER_SPLITS,
    "TOP_K_FIXED": TOP_K_FIXED,
    "N_ITER_SEARCH": N_ITER_SEARCH,
})

orders = pd.read_csv(DATA_DIR / "orders.csv")
op_prior = pd.read_csv(DATA_DIR / "order_products__prior.csv")
op_train = pd.read_csv(DATA_DIR / "order_products__train.csv")
products = pd.read_csv(DATA_DIR / "products.csv")[["product_id", "aisle_id"]]
aisles = pd.read_csv(DATA_DIR / "aisles.csv")[["aisle_id", "aisle"]]

print("orders:", orders.shape)
print("op_prior:", op_prior.shape)
print("op_train:", op_train.shape)
print("products:", products.shape)
print("aisles:", aisles.shape)

{'DATA_DIR': 'data', 'SAMPLE_USERS': 20000, 'OUTER_SPLITS': 3, 'INNER_SPLITS': 3, 'TOP_K_FIXED': 11, 'N_ITER_SEARCH': 12}
orders: (3421083, 7)
op_prior: (32434489, 4)
op_train: (1384617, 4)
products: (49688, 2)
aisles: (134, 2)


In [4]:
# ----------------------------
# Build candidate table and aligned features
# ----------------------------

# Target orders (the labeled final order for users)
target_orders = orders.loc[orders["eval_set"] == "train", ["order_id", "user_id", "order_number"]].copy()
target_orders = target_orders.rename(columns={"order_id": "target_order_id", "order_number": "target_order_number"})

if SAMPLE_USERS is not None:
    rng = np.random.default_rng(RANDOM_STATE)
    sample_uids = rng.choice(target_orders["user_id"].unique(), size=min(SAMPLE_USERS, target_orders["user_id"].nunique()), replace=False)
    target_orders = target_orders[target_orders["user_id"].isin(sample_uids)].copy()

# Prior orders for selected users (attach temporal fields)
prior_orders = orders.loc[orders["eval_set"] == "prior", [
    "order_id", "user_id", "order_number", "days_since_prior_order", "order_dow", "order_hour_of_day"
]].copy()
prior_orders = prior_orders.merge(
    target_orders[["user_id", "target_order_id", "target_order_number"]],
    on="user_id",
    how="inner",
)

# History purchase rows
history = op_prior.merge(
    prior_orders[["order_id", "user_id", "order_number", "days_since_prior_order", "target_order_number"]],
    on="order_id",
    how="inner",
)

# Positive labels from target order
positives = op_train.merge(
    target_orders[["target_order_id", "user_id"]],
    left_on="order_id",
    right_on="target_order_id",
    how="inner",
)
positives = positives[["user_id", "product_id"]].drop_duplicates().assign(label=1)

# Candidate set: user historical products
candidates = history[["user_id", "product_id"]].drop_duplicates().copy()

# User-product aggregates
up_agg = (
    history.groupby(["user_id", "product_id"]).agg(
        up_buy_cnt=("order_id", "size"),
        up_reorder_cnt=("reordered", "sum"),
        up_reorder_ratio=("reordered", "mean"),
        up_last_order=("order_number", "max"),
        up_first_order=("order_number", "min"),
        up_avg_cart_order=("add_to_cart_order", "mean"),
        up_days_since_last_raw=("days_since_prior_order", "mean"),
    )
    .reset_index()
)

# Product aggregates
prod_agg = (
    history.groupby("product_id").agg(
        p_total_purchases=("order_id", "size"),
        p_reorder_ratio=("reordered", "mean"),
        p_avg_cart_order=("add_to_cart_order", "mean"),
        p_unique_users=("user_id", "nunique"),
    )
    .reset_index()
)

# User aggregates
user_order_size = history.groupby(["user_id", "order_id"]).size().rename("basket_size").reset_index()
user_agg = (
    history.groupby("user_id").agg(
        u_total_orders=("order_id", "nunique"),
        u_reorder_ratio=("reordered", "mean"),
        u_unique_products=("product_id", "nunique"),
        u_total_items=("order_id", "size"),
        u_avg_days_between_orders=("days_since_prior_order", "mean"),
    )
    .reset_index()
)
user_agg = user_agg.merge(
    user_order_size.groupby("user_id")["basket_size"].mean().rename("u_avg_basket_size").reset_index(),
    on="user_id",
    how="left",
)

# Merge base modeling table
model_df = candidates.merge(up_agg, on=["user_id", "product_id"], how="left")
model_df = model_df.merge(target_orders[["user_id", "target_order_number"]], on="user_id", how="left")
model_df = model_df.merge(prod_agg, on="product_id", how="left")
model_df = model_df.merge(user_agg, on="user_id", how="left")
model_df = model_df.merge(positives, on=["user_id", "product_id"], how="left")

model_df["label"] = model_df["label"].fillna(0).astype(int)

# Align engineered columns to project naming
model_df["up_orders_since_last"] = (model_df["target_order_number"] - model_df["up_last_order"]).clip(lower=0)
model_df["up_days_since_last"] = (model_df["up_orders_since_last"] * model_df["u_avg_days_between_orders"]).fillna(0)
model_df["up_freq"] = (model_df["up_buy_cnt"] / model_df["u_total_orders"].replace(0, np.nan)).fillna(0)

print("Users:", model_df["user_id"].nunique())
print("Rows:", model_df.shape[0])
print("Positive rate:", round(model_df["label"].mean(), 4))

Users: 20000
Rows: 1294496
Positive rate: 0.0973


In [5]:
# ----------------------------
# Apriori-derived feature: apriori_rule_hits
# Aligned with project description:
# - Build aisle-level rules from historical baskets
# - For each user, use aisles in the most recent historical basket
# - Mark hit if candidate aisle is a rule consequent of any recent-basket aisle
# ----------------------------

prod_aisle = products.merge(aisles, on="aisle_id", how="left")[["product_id", "aisle"]].copy()

history_with_aisle = history.merge(prod_aisle, on="product_id", how="left")

basket_aisles = (
    history_with_aisle.groupby("order_id")["aisle"]
    .apply(lambda x: sorted(set(x.dropna().astype(str))))
    .reset_index(name="aisle_basket")
)
basket_aisles = basket_aisles[basket_aisles["aisle_basket"].str.len() >= 2].copy()

single_counts = {}
pair_counts = {}
for basket in basket_aisles["aisle_basket"]:
    for item in basket:
        single_counts[item] = single_counts.get(item, 0) + 1
    for i in range(len(basket)):
        for j in range(i + 1, len(basket)):
            pair = (basket[i], basket[j])
            pair_counts[pair] = pair_counts.get(pair, 0) + 1


def build_aisle_rules(single_counts, pair_counts, n_baskets, min_support, min_confidence, min_lift):
    rows_local = []
    for (a, b), pair_cnt in pair_counts.items():
        support = pair_cnt / n_baskets
        conf_a_to_b = pair_cnt / single_counts[a]
        conf_b_to_a = pair_cnt / single_counts[b]
        lift_a_to_b = conf_a_to_b / (single_counts[b] / n_baskets)
        lift_b_to_a = conf_b_to_a / (single_counts[a] / n_baskets)

        rows_local.append({"antecedent": a, "consequent": b, "pair_count": pair_cnt, "support": support, "confidence": conf_a_to_b, "lift": lift_a_to_b})
        rows_local.append({"antecedent": b, "consequent": a, "pair_count": pair_cnt, "support": support, "confidence": conf_b_to_a, "lift": lift_b_to_a})

    rules_local = pd.DataFrame(rows_local)
    if len(rules_local) == 0:
        return rules_local

    rules_local = rules_local[
        (rules_local["support"] >= min_support)
        & (rules_local["confidence"] >= min_confidence)
        & (rules_local["lift"] >= min_lift)
    ].copy()

    if len(rules_local) == 0:
        return rules_local

    rules_local = rules_local.sort_values(["lift", "confidence", "pair_count"], ascending=[False, False, False]).reset_index(drop=True)
    return rules_local


rules_filtered = build_aisle_rules(
    single_counts=single_counts,
    pair_counts=pair_counts,
    n_baskets=max(len(basket_aisles), 1),
    min_support=0.01,
    min_confidence=0.30,
    min_lift=1.30,
)

ante_to_cons = {}
if len(rules_filtered) > 0:
    for a, b in rules_filtered[["antecedent", "consequent"]].itertuples(index=False):
        ante_to_cons.setdefault(a, set()).add(b)

# Most recent historical order per user
user_last_order = prior_orders.groupby("user_id")["order_number"].max().rename("last_order_number").reset_index()
last_orders = prior_orders.merge(user_last_order, on="user_id", how="inner")
last_orders = last_orders[last_orders["order_number"] == last_orders["last_order_number"]][["user_id", "order_id"]].drop_duplicates()

last_basket_aisles = (
    last_orders.merge(history_with_aisle[["order_id", "aisle"]], on="order_id", how="left")
    .groupby("user_id")["aisle"]
    .apply(lambda x: set(x.dropna().astype(str)))
)

# For each user, all consequent aisles triggered by their last basket aisles
user_allowed_aisles = {}
for uid, aisle_set in last_basket_aisles.items():
    allowed = set()
    for a in aisle_set:
        allowed.update(ante_to_cons.get(a, set()))
    user_allowed_aisles[uid] = allowed

model_df = model_df.merge(prod_aisle, on="product_id", how="left")
model_df["apriori_rule_hits"] = model_df.apply(
    lambda r: int(r["aisle"] in user_allowed_aisles.get(r["user_id"], set())),
    axis=1,
)
model_df = model_df.drop(columns=["aisle"])

print("Filtered rule count:", len(rules_filtered))
print("apriori_rule_hits mean:", round(model_df["apriori_rule_hits"].mean(), 4))

Filtered rule count: 148
apriori_rule_hits mean: 0.2148


In [7]:
# ----------------------------
# User clusters
# ----------------------------
cluster_source = user_agg[[
    "user_id",
    "u_total_orders",
    "u_avg_days_between_orders",
    "u_avg_basket_size",
    "u_reorder_ratio",
    "u_unique_products",
]].copy()

cluster_feature_cols = [
    "u_total_orders",
    "u_avg_days_between_orders",
    "u_avg_basket_size",
    "u_reorder_ratio",
    "u_unique_products",
]

for c in ["u_total_orders", "u_avg_basket_size", "u_unique_products"]:
    cluster_source[c] = np.log1p(cluster_source[c])

cluster_X = cluster_source[cluster_feature_cols].fillna(0)
cluster_scaler = StandardScaler()
cluster_X_scaled = cluster_scaler.fit_transform(cluster_X)

kmeans_user = KMeans(n_clusters=4, random_state=RANDOM_STATE, n_init=20)
cluster_source["user_cluster"] = kmeans_user.fit_predict(cluster_X_scaled)

# Make this cell rerunnable: remove previously created cluster columns before merge
cluster_cols_to_reset = [
    "user_cluster",
    "user_cluster_1",
    "user_cluster_2",
    "user_cluster_3",
    "cluster_product_target_rate",
]
model_df = model_df.drop(columns=[c for c in cluster_cols_to_reset if c in model_df.columns], errors="ignore")

model_df = model_df.merge(cluster_source[["user_id", "user_cluster"]], on="user_id", how="left")
model_df["user_cluster"] = model_df["user_cluster"].fillna(0).astype(int)

# user_cluster_1/2/3 (drop cluster_0 as baseline)
cluster_dummies = pd.get_dummies(model_df["user_cluster"], prefix="user_cluster", dtype=int)
for c in ["user_cluster_1", "user_cluster_2", "user_cluster_3"]:
    model_df[c] = cluster_dummies[c] if c in cluster_dummies.columns else 0

# IMPORTANT: cluster_product_target_rate is intentionally NOT computed here.
# Computing it on the full model_df would leak labels from the CV test fold into the
# training features. Instead, we add a zero placeholder column and re-fill it inside
# each outer CV fold using only that fold's TRAIN rows (see helpers cell below).
model_df["cluster_product_target_rate"] = 0.0

print(cluster_source["user_cluster"].value_counts().sort_index())
print(model_df[["user_cluster", "user_cluster_1", "user_cluster_2", "user_cluster_3"]].head())

user_cluster
0    5176
1    3584
2    4684
3    6556
Name: count, dtype: int64
   user_cluster  user_cluster_1  user_cluster_2  user_cluster_3
0             0               0               0               0
1             0               0               0               0
2             0               0               0               0
3             0               0               0               0
4             0               0               0               0


In [8]:
aligned_feature_cols = [
    "up_orders_since_last",
    "up_days_since_last",
    "up_freq",
    "up_buy_cnt",
    "up_reorder_ratio",
    "p_reorder_ratio",
    "u_total_orders",
    "cluster_product_target_rate",
    "p_avg_cart_order",
    "u_reorder_ratio",
    "u_unique_products",
    "p_total_purchases",
    "up_first_order",
    "u_avg_days_between_orders",
    "p_unique_users",
    "up_last_order",
    "u_total_items",
    "u_avg_basket_size",
    "up_avg_cart_order",
    "apriori_rule_hits",
    "user_cluster_1",
    "user_cluster_2",
    "user_cluster_3",
]

# Ensure all aligned columns exist
for c in aligned_feature_cols:
    if c not in model_df.columns:
        model_df[c] = 0

model_df[aligned_feature_cols] = model_df[aligned_feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

id_cols = ["user_id", "product_id", "label"]
print("Model df:", model_df.shape)
print("Positive rate:", model_df["label"].mean())

Model df: (1294496, 30)
Positive rate: 0.09734058660667935


In [11]:
# ----------------------------
# Helpers for leakage-safe cluster_product_target_rate
# and for rebuilding Apriori / KMeans features under different hyper-parameters.
# ----------------------------

def fit_cpt_rate(train_df):
    """Fit cluster_product_target_rate from TRAIN rows only."""
    cpt = train_df.groupby(["user_cluster", "product_id"])["label"].mean()
    gm = float(train_df["label"].mean())
    return cpt, gm


def apply_cpt_rate(df, cpt, gm):
    """Map the (user_cluster, product_id) stats into df; unseen combos fall back to gm."""
    idx = pd.MultiIndex.from_arrays([df["user_cluster"].to_numpy(),
                                     df["product_id"].to_numpy()])
    return cpt.reindex(idx).fillna(gm).to_numpy()


# Precompute Apriori co-occurrence tables once (thresholds only affect filtering).
_prod_aisle_cache = prod_aisle
_history_with_aisle_cache = history_with_aisle
_single_counts_cache = single_counts
_pair_counts_cache = pair_counts
_n_baskets_cache = max(len(basket_aisles), 1)

# Precompute each user's last-basket aisles once.
_user_last_basket_aisles = last_basket_aisles


def build_user_allowed(min_support, min_confidence, min_lift):
    """Rebuild the per-user allowed-aisle map for given thresholds."""
    rules = build_aisle_rules(
        _single_counts_cache, _pair_counts_cache, _n_baskets_cache,
        min_support=min_support, min_confidence=min_confidence, min_lift=min_lift,
    )
    ante_map = {}
    if len(rules) > 0:
        for a, b in rules[["antecedent", "consequent"]].itertuples(index=False):
            ante_map.setdefault(a, set()).add(b)
    user_allowed = {}
    for uid, aset in _user_last_basket_aisles.items():
        allowed = set()
        for a in aset:
            allowed.update(ante_map.get(a, set()))
        user_allowed[uid] = allowed
    return user_allowed, int(len(rules))


def attach_apriori_hits(df, user_allowed):
    """Return a numpy array aligned with df's rows containing apriori_rule_hits."""
    tmp = df[["user_id", "product_id"]].merge(_prod_aisle_cache, on="product_id", how="left")
    aisles_arr = tmp["aisle"].to_numpy()
    user_arr = tmp["user_id"].to_numpy()
    out = np.zeros(len(tmp), dtype=np.int8)
    for i in range(len(tmp)):
        s = user_allowed.get(user_arr[i])
        if s is not None and aisles_arr[i] in s:
            out[i] = 1
    return out


def fit_user_clusters(user_agg, n_clusters, random_state=RANDOM_STATE):
    """KMeans clustering on user-level aggregates. Label-free, so can be fit on all users."""
    src = user_agg[[
        "user_id", "u_total_orders", "u_avg_days_between_orders",
        "u_avg_basket_size", "u_reorder_ratio", "u_unique_products",
    ]].copy()
    feats = ["u_total_orders", "u_avg_days_between_orders",
             "u_avg_basket_size", "u_reorder_ratio", "u_unique_products"]
    for c in ["u_total_orders", "u_avg_basket_size", "u_unique_products"]:
        src[c] = np.log1p(src[c])
    X_c = src[feats].fillna(0)
    sc = StandardScaler()
    X_cs = sc.fit_transform(X_c)
    km = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=20)
    src["user_cluster"] = km.fit_predict(X_cs)
    return src[["user_id", "user_cluster"]]


def apply_user_cluster_dummies(df, n_clusters):
    """Return dummy columns user_cluster_1 ... user_cluster_{n_clusters-1} (drop cluster_0)."""
    d = pd.get_dummies(df["user_cluster"], prefix="user_cluster", dtype=int)
    out = pd.DataFrame(index=df.index)
    for k in range(1, n_clusters):
        col = f"user_cluster_{k}"
        out[col] = d[col].to_numpy() if col in d.columns else 0
    return out


def compute_als_features(up_agg, model_df_, factors=32, iterations=15, reg=0.01):
    """Label-free ALS embeddings. Returns (als_dot_array, als_cos_array) aligned to model_df_."""
    try:
        from scipy import sparse
        import implicit
        work = up_agg[["user_id", "product_id", "up_buy_cnt"]].copy()
        u_codes, u_uniques = pd.factorize(work["user_id"], sort=False)
        p_codes, p_uniques = pd.factorize(work["product_id"], sort=False)
        n_users = int(u_codes.max()) + 1
        n_items = int(p_codes.max()) + 1
        ui = sparse.coo_matrix(
            (work["up_buy_cnt"].astype(np.float32).values,
             (u_codes.astype(np.int32), p_codes.astype(np.int32))),
            shape=(n_users, n_items),
        ).tocsr()
        als = implicit.als.AlternatingLeastSquares(
            factors=factors, regularization=reg, iterations=iterations,
            use_gpu=False, random_state=RANDOM_STATE,
        )
        als.fit((ui * 15.0).astype(np.float32))
        uf = als.user_factors
        vf = als.item_factors
        if uf.shape[0] == n_items and vf.shape[0] == n_users:
            uf, vf = vf, uf
        umap = pd.Series(np.arange(len(u_uniques), dtype=np.int32), index=u_uniques)
        imap = pd.Series(np.arange(len(p_uniques), dtype=np.int32), index=p_uniques)
        uc = model_df_["user_id"].map(umap)
        ic = model_df_["product_id"].map(imap)
        valid = uc.notna() & ic.notna()
        dots = np.zeros(len(model_df_), dtype=np.float32)
        coss = np.zeros(len(model_df_), dtype=np.float32)
        u_idx = uc[valid].astype(int).to_numpy()
        i_idx = ic[valid].astype(int).to_numpy()
        uf_sel = uf[u_idx]
        vf_sel = vf[i_idx]
        d = np.sum(uf_sel * vf_sel, axis=1).astype(np.float32)
        denom = np.maximum(np.linalg.norm(uf_sel, axis=1) * np.linalg.norm(vf_sel, axis=1), 1e-8)
        c = (d / denom).astype(np.float32)
        mask = valid.to_numpy()
        dots[mask] = d
        coss[mask] = c
        return dots, coss, "enabled"
    except Exception as exc:
        return (np.zeros(len(model_df_), dtype=np.float32),
                np.zeros(len(model_df_), dtype=np.float32),
                f"failed: {exc}")


print("Helpers defined: fit_cpt_rate, apply_cpt_rate, build_user_allowed, "
      "attach_apriori_hits, fit_user_clusters, apply_user_cluster_dummies, compute_als_features")


Helpers defined: fit_cpt_rate, apply_cpt_rate, build_user_allowed, attach_apriori_hits, fit_user_clusters, apply_user_cluster_dummies, compute_als_features


In [12]:
# ----------------------------
# Joint tuning: Apriori thresholds x KMeans n_clusters
# ----------------------------
# Rationale:
# - Apriori rules decide which candidates are "co-occurrence friendly";
#   KMeans user segmentation decides which cluster averages each row sees.
# - The two features interact: a looser Apriori means more rule hits, so the
#   signal a cluster can add is different. Tuning them jointly is more honest
#   than fixing one and tuning the other.
# - We use a lightweight proxy (LogReg under GroupKFold(3) AUC) per grid point
#   to keep the search tractable.
# - cluster_product_target_rate is (re)fit from TRAIN rows only in each fold.

import itertools
from sklearn.utils import check_random_state

joint_grid = {
    "min_support":    [0.005, 0.01, 0.02],
    "min_confidence": [0.20, 0.30, 0.40],
    "min_lift":       [1.10, 1.30, 1.50],
    "n_clusters":     [3, 4, 5, 6],
}

all_combos = list(itertools.product(
    joint_grid["min_support"], joint_grid["min_confidence"],
    joint_grid["min_lift"], joint_grid["n_clusters"],
))
N_JOINT_TRIALS = 10
rs = check_random_state(RANDOM_STATE)
chosen = rs.choice(len(all_combos), size=min(N_JOINT_TRIALS, len(all_combos)), replace=False)
joint_trials = [all_combos[i] for i in chosen]

# Columns that stay fixed across the joint search (the 19 that do not depend on
# Apriori/KMeans). We then append apriori_rule_hits, user_cluster_* dummies and
# cluster_product_target_rate according to each trial.
fixed_base_cols = [c for c in aligned_feature_cols
                   if c not in ("apriori_rule_hits",
                                "cluster_product_target_rate",
                                "user_cluster_1", "user_cluster_2", "user_cluster_3")]

proxy_cv = GroupKFold(n_splits=3)
joint_rows = []

for sup, cnf, lft, kc in joint_trials:
    df_eval = model_df[["user_id", "product_id", "label"] + fixed_base_cols].copy()

    user_allowed, n_rules = build_user_allowed(min_support=sup,
                                               min_confidence=cnf,
                                               min_lift=lft)
    df_eval["apriori_rule_hits"] = attach_apriori_hits(df_eval, user_allowed)

    uc = fit_user_clusters(user_agg, n_clusters=kc)
    df_eval = df_eval.merge(uc, on="user_id", how="left")
    df_eval["user_cluster"] = df_eval["user_cluster"].fillna(0).astype(int)
    dummy_cols = [f"user_cluster_{k}" for k in range(1, kc)]
    dummies = apply_user_cluster_dummies(df_eval, n_clusters=kc)
    for col in dummy_cols:
        df_eval[col] = dummies[col].to_numpy()

    feat_cols = (fixed_base_cols + ["apriori_rule_hits"] + dummy_cols
                 + ["cluster_product_target_rate"])

    y_ev = df_eval["label"].astype(int).to_numpy()
    g_ev = df_eval["user_id"].to_numpy()

    aucs = []
    for tr_idx, te_idx in proxy_cv.split(df_eval, y_ev, groups=g_ev):
        tr_df = df_eval.iloc[tr_idx]
        cpt, gm = fit_cpt_rate(tr_df)

        cpt_all = apply_cpt_rate(df_eval, cpt, gm)
        df_eval_local = df_eval.copy()
        df_eval_local["cluster_product_target_rate"] = cpt_all

        X_mat = df_eval_local[feat_cols].replace([np.inf, -np.inf], np.nan).fillna(0).to_numpy()
        y_tr_ev, y_te_ev = y_ev[tr_idx], y_ev[te_idx]

        proxy = Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                solver="saga", class_weight="balanced",
                C=0.3, max_iter=200, random_state=RANDOM_STATE, n_jobs=-1,
            )),
        ])
        proxy.fit(X_mat[tr_idx], y_tr_ev)
        p = proxy.predict_proba(X_mat[te_idx])[:, 1]
        aucs.append(roc_auc_score(y_te_ev, p))

    mean_auc = float(np.mean(aucs))
    joint_rows.append({
        "min_support": sup, "min_confidence": cnf, "min_lift": lft,
        "n_clusters": kc, "n_rules": n_rules, "mean_auc": mean_auc,
    })
    print(f"sup={sup:.3f} conf={cnf:.2f} lift={lft:.2f} K={kc} | "
          f"rules={n_rules:4d} | proxy_auc={mean_auc:.4f}")

joint_results_df = (pd.DataFrame(joint_rows)
                    .sort_values("mean_auc", ascending=False)
                    .reset_index(drop=True))
best_joint = joint_results_df.iloc[0].to_dict()
print("\nBest joint params:", best_joint)
joint_results_df


sup=0.020 conf=0.20 lift=1.30 K=4 | rules= 150 | proxy_auc=0.7927
sup=0.005 conf=0.20 lift=1.50 K=5 | rules= 155 | proxy_auc=0.7888
sup=0.005 conf=0.20 lift=1.30 K=3 | rules= 362 | proxy_auc=0.7974
sup=0.020 conf=0.20 lift=1.50 K=6 | rules=  51 | proxy_auc=0.7867
sup=0.010 conf=0.40 lift=1.10 K=5 | rules= 180 | proxy_auc=0.7887
sup=0.010 conf=0.40 lift=1.30 K=6 | rules=  63 | proxy_auc=0.7867
sup=0.005 conf=0.40 lift=1.30 K=5 | rules=  70 | proxy_auc=0.7887
sup=0.010 conf=0.20 lift=1.50 K=4 | rules= 106 | proxy_auc=0.7927
sup=0.020 conf=0.30 lift=1.50 K=6 | rules=  27 | proxy_auc=0.7867
sup=0.005 conf=0.20 lift=1.50 K=6 | rules= 155 | proxy_auc=0.7867

Best joint params: {'min_support': 0.005, 'min_confidence': 0.2, 'min_lift': 1.3, 'n_clusters': 3.0, 'n_rules': 362.0, 'mean_auc': 0.7974167854251647}


,min_support,min_confidence,min_lift,n_clusters,n_rules,mean_auc
0,0.005,0.2,1.3,3,362,0.797417
1,0.010,0.2,1.5,4,106,0.792669
2,0.020,0.2,1.3,4,150,0.792668
3,0.005,0.2,1.5,5,155,0.788756
4,0.005,0.4,1.3,5,70,0.788699
5,0.010,0.4,1.1,5,180,0.788676
6,0.005,0.2,1.5,6,155,0.786698
7,0.020,0.2,1.5,6,51,0.786688
8,0.020,0.3,1.5,6,27,0.786665
9,0.010,0.4,1.3,6,63,0.786650


In [13]:
# ----------------------------
# Apply the best (Apriori + KMeans) configuration back onto model_df,
# compute ALS features, and define the two feature sets for model tuning.
# ----------------------------

best_sup = float(best_joint["min_support"])
best_cnf = float(best_joint["min_confidence"])
best_lft = float(best_joint["min_lift"])
best_k = int(best_joint["n_clusters"])

# Rebuild Apriori rule hits on the full model_df with the winning thresholds
best_user_allowed, n_rules_best = build_user_allowed(
    min_support=best_sup, min_confidence=best_cnf, min_lift=best_lft,
)
model_df["apriori_rule_hits"] = attach_apriori_hits(model_df, best_user_allowed)

# Rebuild user clusters with the winning K, refresh dummy columns
uc_best = fit_user_clusters(user_agg, n_clusters=best_k)
drop_cols = [c for c in model_df.columns if c.startswith("user_cluster")]
model_df = model_df.drop(columns=drop_cols, errors="ignore")
model_df = model_df.merge(uc_best, on="user_id", how="left")
model_df["user_cluster"] = model_df["user_cluster"].fillna(0).astype(int)
cluster_dummy_cols = [f"user_cluster_{k}" for k in range(1, best_k)]
best_dummies = apply_user_cluster_dummies(model_df, n_clusters=best_k)
for col in cluster_dummy_cols:
    model_df[col] = best_dummies[col].to_numpy()

# cluster_product_target_rate stays as a placeholder -- it will be refilled
# from the TRAIN rows inside every outer CV fold below.
model_df["cluster_product_target_rate"] = 0.0

# ALS features (label-free; computed once across all users for speed)
als_dots, als_coss, als_status = compute_als_features(up_agg, model_df)
model_df["als_dot"] = als_dots
model_df["als_cos"] = als_coss
print("ALS status:", als_status, "| rules used:", n_rules_best, "| best K:", best_k)

# Build the two feature views we will tune on
base_feature_cols_dynamic = [
    c for c in aligned_feature_cols
    if c not in ("user_cluster_1", "user_cluster_2", "user_cluster_3")
] + cluster_dummy_cols

feature_sets = {
    "no_als":   list(base_feature_cols_dynamic),
    "with_als": list(base_feature_cols_dynamic) + ["als_dot", "als_cos"],
}
print("feature_sets sizes:", {k: len(v) for k, v in feature_sets.items()})


ALS status: failed: No module named 'implicit' | rules used: 362 | best K: 3
feature_sets sizes: {'no_als': 22, 'with_als': 24}


In [14]:
def eval_row_level(y_true, y_prob):
    eps = 1e-15
    y_prob_clip = np.clip(y_prob, eps, 1 - eps)
    out = {
        "auc": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan,
        "pr_auc": average_precision_score(y_true, y_prob),
        "logloss": log_loss(y_true, y_prob_clip),
    }
    return out


def eval_order_topk(df_eval, prob_col="prob", k=11, label_col="label"):
    ranked = df_eval.sort_values(["user_id", prob_col, "product_id"], ascending=[True, False, True]).copy()
    ranked["rank_within_user"] = ranked.groupby("user_id").cumcount() + 1
    ranked["pred_topk"] = (ranked["rank_within_user"] <= k).astype(int)

    order_precision, order_recall, order_f1, order_hit = [], [], [], []
    for _, g in ranked.groupby("user_id"):
        y_t = g[label_col].to_numpy()
        y_p = g["pred_topk"].to_numpy()
        order_precision.append(precision_score(y_t, y_p, zero_division=0))
        order_recall.append(recall_score(y_t, y_p, zero_division=0))
        order_f1.append(f1_score(y_t, y_p, zero_division=0))
        order_hit.append(float((g.loc[g["pred_topk"] == 1, label_col].sum() > 0)))

    return {
        f"precision@{k}": float(np.mean(order_precision)),
        f"recall@{k}": float(np.mean(order_recall)),
        f"f1@{k}": float(np.mean(order_f1)),
        f"hit@{k}": float(np.mean(order_hit)),
    }

# pred_k = round(u_avg_basket_size), clipped at >=1
def make_dynamic_k_map(df_input):
    user_k_df = df_input.groupby("user_id")["u_avg_basket_size"].first().reset_index()
    user_k_df["pred_k"] = np.rint(user_k_df["u_avg_basket_size"]).astype(int).clip(lower=1)
    return dict(zip(user_k_df["user_id"], user_k_df["pred_k"]))


def eval_order_dynamic_k(df_eval, prob_col, k_pred_map, label_col="label"):
    ranked = df_eval.sort_values(["user_id", prob_col, "product_id"], ascending=[True, False, True]).copy()
    ranked["rank_within_user"] = ranked.groupby("user_id").cumcount() + 1
    ranked["pred_k"] = ranked["user_id"].map(k_pred_map).fillna(TOP_K_FIXED).astype(int)
    ranked["pred_k"] = ranked["pred_k"].clip(lower=1)
    ranked["pred_topk_dynamic"] = (ranked["rank_within_user"] <= ranked["pred_k"]).astype(int)

    order_precision, order_recall, order_f1, order_hit, avg_pred_k = [], [], [], [], []
    for _, g in ranked.groupby("user_id"):
        y_t = g[label_col].to_numpy()
        y_p = g["pred_topk_dynamic"].to_numpy()
        order_precision.append(precision_score(y_t, y_p, zero_division=0))
        order_recall.append(recall_score(y_t, y_p, zero_division=0))
        order_f1.append(f1_score(y_t, y_p, zero_division=0))
        order_hit.append(float((g.loc[g["pred_topk_dynamic"] == 1, label_col].sum() > 0)))
        avg_pred_k.append(float(g["pred_k"].iloc[0]))

    return {
        "precision@dynamic_k": float(np.mean(order_precision)),
        "recall@dynamic_k": float(np.mean(order_recall)),
        "f1@dynamic_k": float(np.mean(order_f1)),
        "hit@dynamic_k": float(np.mean(order_hit)),
        "avg_pred_k": float(np.mean(avg_pred_k)),
    }

In [15]:
y = model_df["label"].astype(int).to_numpy()
groups = model_df["user_id"].to_numpy()

outer_cv = GroupKFold(n_splits=OUTER_SPLITS)
inner_cv = GroupKFold(n_splits=INNER_SPLITS)


def build_fold_feature_matrix(tr_idx, feature_cols):
    """Return a full-sized numpy X matrix whose cluster_product_target_rate
    column has been fit from TRAIN rows only. All other columns come from
    model_df as-is. Training rows use train-fold cpt_rate, test rows use
    the same train-fold statistics (this is the leakage-safe protocol)."""
    tr_df = model_df.iloc[tr_idx]
    cpt, gm = fit_cpt_rate(tr_df)
    cpt_vec = apply_cpt_rate(model_df, cpt, gm)
    X_full = model_df[feature_cols].copy()
    if "cluster_product_target_rate" in feature_cols:
        X_full["cluster_product_target_rate"] = cpt_vec
    X_full = X_full.replace([np.inf, -np.inf], np.nan).fillna(0)
    return X_full.to_numpy()


logreg_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        solver="saga",
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=0,
    )),
])

logreg_param_dist = {
    "model__C": [0.03, 0.1, 0.3, 1.0, 3.0],
    "model__max_iter": [150, 250, 400, 600],
}

if HAS_LIGHTGBM:
    lgb_model = lgb.LGBMClassifier(
        objective="binary",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1,
    )
    lgb_param_dist = {
        "num_leaves": [31, 63, 127],
        "learning_rate": [0.02, 0.03, 0.05, 0.1],
        "n_estimators": [250, 400, 600],
        "min_child_samples": [20, 30, 50, 100],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
    }
else:
    raise RuntimeError("LightGBM is required for this notebook. Please install lightgbm first.")

model_spaces = {
    "LogisticRegression_SAGA": (logreg_pipe, logreg_param_dist),
    "LightGBM": (lgb_model, lgb_param_dist),
}

print("Users:", len(np.unique(groups)),
      "| feature_sets:", {k: len(v) for k, v in feature_sets.items()})

Users: 20000 | feature_sets: {'no_als': 22, 'with_als': 24}


In [16]:
rows = []

for fs_name, feat_cols in feature_sets.items():
    print(f"\n########## Feature set: {fs_name} ({len(feat_cols)} cols) ##########")

    for model_name, (estimator, param_dist) in model_spaces.items():
        print(f"\n===== Tuning {model_name} | {fs_name} =====")

        for fold_id, (tr_idx, te_idx) in enumerate(outer_cv.split(model_df, y, groups=groups), start=1):
            X_full = build_fold_feature_matrix(tr_idx, feat_cols)
            X_tr, X_te = X_full[tr_idx], X_full[te_idx]
            y_tr, y_te = y[tr_idx], y[te_idx]
            g_tr = groups[tr_idx]

            search = RandomizedSearchCV(
                estimator=estimator,
                param_distributions=param_dist,
                n_iter=N_ITER_SEARCH,
                scoring="roc_auc",
                cv=inner_cv,
                random_state=RANDOM_STATE,
                n_jobs=-1,
                refit=True,
                verbose=0,
            )
            search.fit(X_tr, y_tr, groups=g_tr)

            best_est = search.best_estimator_
            y_prob = best_est.predict_proba(X_te)[:, 1]

            fold_df = model_df.iloc[te_idx][["user_id", "product_id", "label", "u_avg_basket_size"]].copy()
            fold_df["prob"] = y_prob
            k_map = make_dynamic_k_map(fold_df)

            row = {
                "feature_set": fs_name,
                "model": model_name,
                "outer_fold": fold_id,
                "inner_best_score_auc": float(search.best_score_),
                "best_params": str(search.best_params_),
            }
            row.update(eval_row_level(y_te, y_prob))
            row.update(eval_order_topk(fold_df, prob_col="prob", k=TOP_K_FIXED, label_col="label"))
            row.update(eval_order_dynamic_k(fold_df, prob_col="prob", k_pred_map=k_map, label_col="label"))

            rows.append(row)
            print(
                f"  fold {fold_id} | auc={row['auc']:.4f} | pr_auc={row['pr_auc']:.4f} | "
                f"f1@11={row['f1@11']:.4f} | f1@dynamic_k={row['f1@dynamic_k']:.4f}"
            )

results_df = pd.DataFrame(rows)
results_df


########## Feature set: no_als (22 cols) ##########

===== Tuning LogisticRegression_SAGA | no_als =====
  fold 1 | auc=0.7977 | pr_auc=0.2982 | f1@11=0.3064 | f1@dynamic_k=0.3219
  fold 2 | auc=0.7969 | pr_auc=0.3025 | f1@11=0.3076 | f1@dynamic_k=0.3245
  fold 3 | auc=0.7978 | pr_auc=0.3013 | f1@11=0.3111 | f1@dynamic_k=0.3235

===== Tuning LightGBM | no_als =====
  fold 1 | auc=0.7549 | pr_auc=0.3286 | f1@11=0.3128 | f1@dynamic_k=0.3363
  fold 2 | auc=0.7549 | pr_auc=0.3317 | f1@11=0.3154 | f1@dynamic_k=0.3372
  fold 3 | auc=0.7634 | pr_auc=0.3371 | f1@11=0.3185 | f1@dynamic_k=0.3372

########## Feature set: with_als (24 cols) ##########

===== Tuning LogisticRegression_SAGA | with_als =====
  fold 1 | auc=0.7977 | pr_auc=0.2982 | f1@11=0.3064 | f1@dynamic_k=0.3219
  fold 2 | auc=0.7969 | pr_auc=0.3025 | f1@11=0.3076 | f1@dynamic_k=0.3245
  fold 3 | auc=0.7978 | pr_auc=0.3013 | f1@11=0.3111 | f1@dynamic_k=0.3235

===== Tuning LightGBM | with_als =====
  fold 1 | auc=0.7549 | pr_auc=

,feature_set,model,outer_fold,inner_best_score_auc,best_params,auc,pr_auc,logloss,precision@11,recall@11,f1@11,hit@11,precision@dynamic_k,recall@dynamic_k,f1@dynamic_k,hit@dynamic_k,avg_pred_k
0,no_als,LogisticRegression_SAGA,1,0.860473,"{'model__max_iter': 150, 'model__C': 0.03}",0.797667,0.298237,0.518989,0.264653,0.512298,0.306389,0.858557,0.286189,0.442236,0.321926,0.804110,9.987851
1,no_als,LogisticRegression_SAGA,2,0.860896,"{'model__max_iter': 150, 'model__C': 0.03}",0.796866,0.302486,0.516189,0.266767,0.512843,0.307615,0.862157,0.289642,0.444516,0.324539,0.810259,9.963702
2,no_als,LogisticRegression_SAGA,3,0.860940,"{'model__max_iter': 150, 'model__C': 0.03}",0.797810,0.301260,0.521315,0.267268,0.522334,0.311132,0.863636,0.287633,0.449204,0.323511,0.812631,9.980048
3,no_als,LightGBM,1,0.875429,"{'subsample': 0.8, 'num_leaves': 63, 'n_estima...",0.754898,0.328641,0.336914,0.272207,0.515934,0.312766,0.858707,0.299620,0.460747,0.336267,0.815359,9.987851
4,no_als,LightGBM,2,0.876004,"{'subsample': 0.8, 'num_leaves': 127, 'n_estim...",0.754854,0.331703,0.311602,0.275535,0.519710,0.315359,0.862457,0.301235,0.461876,0.337242,0.821359,9.963702
5,no_als,LightGBM,3,0.875294,"{'subsample': 0.8, 'num_leaves': 63, 'n_estima...",0.763408,0.337083,0.329391,0.275737,0.527454,0.318519,0.861386,0.301117,0.465393,0.337222,0.818182,9.980048
6,with_als,LogisticRegression_SAGA,1,0.860473,"{'model__max_iter': 150, 'model__C': 0.03}",0.797667,0.298237,0.518989,0.264653,0.512298,0.306389,0.858557,0.286189,0.442236,0.321926,0.804110,9.987851
7,with_als,LogisticRegression_SAGA,2,0.860896,"{'model__max_iter': 150, 'model__C': 0.03}",0.796866,0.302486,0.516189,0.266767,0.512843,0.307615,0.862157,0.289642,0.444516,0.324539,0.810259,9.963702
8,with_als,LogisticRegression_SAGA,3,0.860940,"{'model__max_iter': 150, 'model__C': 0.03}",0.797810,0.301260,0.521315,0.267268,0.522334,0.311132,0.863636,0.287633,0.449204,0.323511,0.812631,9.980048
9,with_als,LightGBM,1,0.875429,"{'subsample': 0.8, 'num_leaves': 63, 'n_estima...",0.754898,0.328641,0.336914,0.272207,0.515934,0.312766,0.858707,0.299620,0.460747,0.336267,0.815359,9.987851


In [17]:
summary_df = (
    results_df.groupby(["model", "feature_set"], as_index=False)
    .agg(
        auc_mean=("auc", "mean"),
        auc_std=("auc", "std"),
        pr_auc_mean=("pr_auc", "mean"),
        logloss_mean=("logloss", "mean"),
        p11_mean=("precision@11", "mean"),
        r11_mean=("recall@11", "mean"),
        f11_mean=("f1@11", "mean"),
        hit11_mean=("hit@11", "mean"),
        pdk_mean=("precision@dynamic_k", "mean"),
        rdk_mean=("recall@dynamic_k", "mean"),
        fdk_mean=("f1@dynamic_k", "mean"),
        hitdk_mean=("hit@dynamic_k", "mean"),
        avg_pred_k=("avg_pred_k", "mean"),
    )
    .sort_values(["feature_set", "auc_mean"], ascending=[True, False])
)

print("=== Summary across outer folds by (model, feature_set) ===")
summary_df

=== Summary across outer folds by (model, feature_set) ===


,model,feature_set,auc_mean,auc_std,pr_auc_mean,logloss_mean,p11_mean,r11_mean,f11_mean,hit11_mean,pdk_mean,rdk_mean,fdk_mean,hitdk_mean,avg_pred_k
2,LogisticRegression_SAGA,no_als,0.797448,0.000508,0.300661,0.518831,0.266229,0.515825,0.308379,0.86145,0.287822,0.445319,0.323326,0.8090,9.9772
0,LightGBM,no_als,0.757720,0.004926,0.332476,0.325969,0.274493,0.521033,0.315548,0.86085,0.300657,0.462672,0.336911,0.8183,9.9772
3,LogisticRegression_SAGA,with_als,0.797448,0.000508,0.300661,0.518831,0.266229,0.515825,0.308379,0.86145,0.287822,0.445319,0.323326,0.8090,9.9772
1,LightGBM,with_als,0.757720,0.004926,0.332476,0.325969,0.274493,0.521033,0.315548,0.86085,0.300657,0.462672,0.336911,0.8183,9.9772


In [18]:
# ----------------------------
# MLP tuning
# Rationale:
# - LightGBM > LogReg by a clear margin, so MLP should be regularized and stable first.
# ----------------------------

from sklearn.neural_network import MLPClassifier

mlp_pipe = Pipeline([
    ("scaler", StandardScaler()),
    (
        "model",
        MLPClassifier(
            random_state=RANDOM_STATE,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=8,
            max_iter=80,
            verbose=False,
        ),
    ),
])

mlp_param_dist = {
    # Moderate-size nets to control runtime and overfit risk
    "model__hidden_layer_sizes": [(64,), (128,), (128, 64)],
    # Stronger regularization range due to class imbalance/noisy negatives
    "model__alpha": [1e-4, 3e-4, 1e-3, 3e-3],
    # Conservative learning rates for stable convergence
    "model__learning_rate_init": [3e-4, 6e-4, 1e-3],
    "model__batch_size": [256, 512, 1024],
}

mlp_rows = []

for fs_name, feat_cols in feature_sets.items():
    print(f"\n########## MLP feature set: {fs_name} ({len(feat_cols)} cols) ##########")

    for fold_id, (tr_idx, te_idx) in enumerate(outer_cv.split(model_df, y, groups=groups), start=1):
        X_full = build_fold_feature_matrix(tr_idx, feat_cols)
        X_tr, X_te = X_full[tr_idx], X_full[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]
        g_tr = groups[tr_idx]

        mlp_search = RandomizedSearchCV(
            estimator=mlp_pipe,
            param_distributions=mlp_param_dist,
            n_iter=10,
            scoring="roc_auc",
            cv=inner_cv,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            refit=True,
            verbose=0,
        )
        mlp_search.fit(X_tr, y_tr, groups=g_tr)

        mlp_best = mlp_search.best_estimator_
        mlp_prob = mlp_best.predict_proba(X_te)[:, 1]

        fold_df = model_df.iloc[te_idx][["user_id", "product_id", "label", "u_avg_basket_size"]].copy()
        fold_df["prob"] = mlp_prob
        k_map = make_dynamic_k_map(fold_df)

        row = {
            "feature_set": fs_name,
            "model": "MLP",
            "outer_fold": fold_id,
            "inner_best_score_auc": float(mlp_search.best_score_),
            "best_params": str(mlp_search.best_params_),
        }
        row.update(eval_row_level(y_te, mlp_prob))
        row.update(eval_order_topk(fold_df, prob_col="prob", k=TOP_K_FIXED, label_col="label"))
        row.update(eval_order_dynamic_k(fold_df, prob_col="prob", k_pred_map=k_map, label_col="label"))

        mlp_rows.append(row)
        print(
            f"  fold {fold_id} | auc={row['auc']:.4f} | pr_auc={row['pr_auc']:.4f} | "
            f"f1@11={row['f1@11']:.4f} | f1@dynamic_k={row['f1@dynamic_k']:.4f}"
        )

mlp_results_df = pd.DataFrame(mlp_rows)
mlp_results_df


########## MLP feature set: no_als (22 cols) ##########
  fold 1 | auc=0.7650 | pr_auc=0.3291 | f1@11=0.3144 | f1@dynamic_k=0.3349
  fold 2 | auc=0.7671 | pr_auc=0.3332 | f1@11=0.3163 | f1@dynamic_k=0.3376
  fold 3 | auc=0.7650 | pr_auc=0.3316 | f1@11=0.3184 | f1@dynamic_k=0.3369

########## MLP feature set: with_als (24 cols) ##########
  fold 1 | auc=0.7656 | pr_auc=0.3279 | f1@11=0.3132 | f1@dynamic_k=0.3338
  fold 2 | auc=0.7631 | pr_auc=0.3303 | f1@11=0.3171 | f1@dynamic_k=0.3382
  fold 3 | auc=0.7733 | pr_auc=0.3322 | f1@11=0.3194 | f1@dynamic_k=0.3368


,feature_set,model,outer_fold,inner_best_score_auc,best_params,auc,pr_auc,logloss,precision@11,recall@11,f1@11,hit@11,precision@dynamic_k,recall@dynamic_k,f1@dynamic_k,hit@dynamic_k,avg_pred_k
0,no_als,MLP,1,0.875692,"{'model__learning_rate_init': 0.0006, 'model__...",0.764993,0.329107,0.321581,0.273325,0.519528,0.314448,0.861107,0.298380,0.458535,0.334860,0.813709,9.987851
1,no_als,MLP,2,0.876312,"{'model__learning_rate_init': 0.001, 'model__h...",0.767145,0.333223,0.316182,0.276353,0.521696,0.316324,0.862907,0.301777,0.461699,0.337616,0.819409,9.963702
2,no_als,MLP,3,0.876012,"{'model__learning_rate_init': 0.001, 'model__h...",0.764960,0.331628,0.348810,0.275682,0.527005,0.318429,0.861686,0.300644,0.465271,0.336927,0.817732,9.980048
3,with_als,MLP,1,0.875575,"{'model__learning_rate_init': 0.001, 'model__h...",0.765587,0.327850,0.304981,0.272371,0.516999,0.313193,0.859157,0.297443,0.456657,0.333795,0.811909,9.987851
4,with_als,MLP,2,0.876637,"{'model__learning_rate_init': 0.001, 'model__h...",0.763141,0.330347,0.359672,0.277225,0.521826,0.317127,0.862907,0.302017,0.462546,0.338162,0.821359,9.963702
5,with_als,MLP,3,0.875662,"{'model__learning_rate_init': 0.001, 'model__h...",0.773303,0.332188,0.295170,0.276241,0.529174,0.319364,0.862286,0.300551,0.465593,0.336802,0.818632,9.980048


In [19]:
all_results_df = pd.concat([results_df, mlp_results_df], ignore_index=True)

all_summary_df = (
    all_results_df.groupby(["model", "feature_set"], as_index=False)
    .agg(
        auc_mean=("auc", "mean"),
        auc_std=("auc", "std"),
        pr_auc_mean=("pr_auc", "mean"),
        logloss_mean=("logloss", "mean"),
        p11_mean=("precision@11", "mean"),
        r11_mean=("recall@11", "mean"),
        f11_mean=("f1@11", "mean"),
        hit11_mean=("hit@11", "mean"),
        pdk_mean=("precision@dynamic_k", "mean"),
        rdk_mean=("recall@dynamic_k", "mean"),
        fdk_mean=("f1@dynamic_k", "mean"),
        hitdk_mean=("hit@dynamic_k", "mean"),
        avg_pred_k=("avg_pred_k", "mean"),
    )
    .sort_values(["feature_set", "auc_mean"], ascending=[True, False])
)

# For each (model, feature_set), the best params are the outer-fold params
# with the highest inner_best_score_auc (the fold where inner CV was most confident).
def _pick_best_params(grp):
    best_row = grp.sort_values("inner_best_score_auc", ascending=False).iloc[0]
    return pd.Series({
        "inner_best_score_auc": float(best_row["inner_best_score_auc"]),
        "best_params": best_row["best_params"],
    })

best_params_per_group = (
    all_results_df.groupby(["model", "feature_set"])
    .apply(_pick_best_params)
    .reset_index()
    .sort_values(["model", "feature_set"])
)

print("=== Joint tuning winner (Apriori + KMeans) ===")
print(f"min_support={best_sup}, min_confidence={best_cnf}, min_lift={best_lft}, n_clusters={best_k}")

print("\n=== Summary by (model, feature_set) ===")
print(all_summary_df.to_string(index=False))

print("\n=== Best params per (model, feature_set) ===")
print(best_params_per_group.to_string(index=False))

all_summary_df

=== Joint tuning winner (Apriori + KMeans) ===
min_support=0.005, min_confidence=0.2, min_lift=1.3, n_clusters=3

=== Summary by (model, feature_set) ===
                  model feature_set  auc_mean  auc_std  pr_auc_mean  logloss_mean  p11_mean  r11_mean  f11_mean  hit11_mean  pdk_mean  rdk_mean  fdk_mean  hitdk_mean  avg_pred_k
LogisticRegression_SAGA      no_als  0.797448 0.000508     0.300661      0.518831  0.266229  0.515825  0.308379     0.86145  0.287822  0.445319  0.323326     0.80900      9.9772
                    MLP      no_als  0.765699 0.001252     0.331319      0.328858  0.275120  0.522743  0.316400     0.86190  0.300267  0.461835  0.336468     0.81695      9.9772
               LightGBM      no_als  0.757720 0.004926     0.332476      0.325969  0.274493  0.521033  0.315548     0.86085  0.300657  0.462672  0.336911     0.81830      9.9772
LogisticRegression_SAGA    with_als  0.797448 0.000508     0.300661      0.518831  0.266229  0.515825  0.308379     0.86145  0.287822 

,model,feature_set,auc_mean,auc_std,pr_auc_mean,logloss_mean,p11_mean,r11_mean,f11_mean,hit11_mean,pdk_mean,rdk_mean,fdk_mean,hitdk_mean,avg_pred_k
2,LogisticRegression_SAGA,no_als,0.797448,0.000508,0.300661,0.518831,0.266229,0.515825,0.308379,0.86145,0.287822,0.445319,0.323326,0.80900,9.9772
4,MLP,no_als,0.765699,0.001252,0.331319,0.328858,0.275120,0.522743,0.316400,0.86190,0.300267,0.461835,0.336468,0.81695,9.9772
0,LightGBM,no_als,0.757720,0.004926,0.332476,0.325969,0.274493,0.521033,0.315548,0.86085,0.300657,0.462672,0.336911,0.81830,9.9772
3,LogisticRegression_SAGA,with_als,0.797448,0.000508,0.300661,0.518831,0.266229,0.515825,0.308379,0.86145,0.287822,0.445319,0.323326,0.80900,9.9772
5,MLP,with_als,0.767344,0.005304,0.330128,0.319941,0.275279,0.522666,0.316561,0.86145,0.300004,0.461599,0.336253,0.81730,9.9772
1,LightGBM,with_als,0.757720,0.004926,0.332476,0.325969,0.274493,0.521033,0.315548,0.86085,0.300657,0.462672,0.336911,0.81830,9.9772


`Users`：用户数。
`Rows`：候选 `(user_id, product_id)` 总行数。
`Positive rate`：正样本占比（类别不平衡程度）。

`Filtered rule count`：Apriori 规则条数（阈值过滤后）。
`apriori_rule_hits mean`：命中规则的候选占比。

`inner_best_score_auc`：内层调参最优 AUC。
`auc / pr_auc`：越大越好。
`logloss`：越小越好。
`precision@11 / recall@11 / f1@11 / hit@11`：固定推荐 11 个时的效果。
`precision@dynamic_k / recall@dynamic_k / f1@dynamic_k / hit@dynamic_k`：动态推荐长度时的效果。
`avg_pred_k`：动态推荐平均长度（应接近用户平均篮子大小）。

`auc_mean`、`pr_auc_mean`、`f11_mean`、`fdk_mean` 比较模型。
LightGBM 在这些核心指标上都高于 Logistic Regression。

`mlp_results_df`含义与 `results_df` 一样，只是模型变成 MLP。
选型优先看：`pr_auc_mean` + `fdk_mean`，再 `logloss_mean`。